# Naïve Bayes Implementation.

In [1]:
documents = [
    "Free money now",
    "Hi mom how are you",
    "Lowest price for your meds",
    "Are we still on for dinner",
    "Win a free iPhone today",
    "Lets catch up tomorrow at the office",
    "Meeting at 3 PM tomorrow",
    "Get 50 percent off limited time",
    "Team meeting in the office",
    "Click here for prizes",
    "Can you send the report"
]

labels = ["SPAM","HAM","SPAM","HAM","SPAM","HAM","HAM","SPAM","HAM","SPAM","HAM"]

In [2]:
import re
from collections import defaultdict, Counter

def tokenize(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9 ]', '', text)
    return text.split()

In [3]:
def build_vocab(docs):
    vocab = set()
    for doc in docs:
        vocab.update(tokenize(doc))
    return sorted(vocab)

vocab = build_vocab(documents)
vocab_size = len(vocab)
vocab_size

45

In [4]:
from collections import Counter

class_counts = Counter(labels)
total_docs = len(labels)

priors = {c: class_counts[c]/total_docs for c in class_counts}
priors

{'SPAM': 0.45454545454545453, 'HAM': 0.5454545454545454}

In [5]:
word_counts = {
    "SPAM": Counter(),
    "HAM": Counter()
}

total_words = {"SPAM": 0, "HAM": 0}

for doc, label in zip(documents, labels):
    tokens = tokenize(doc)
    word_counts[label].update(tokens)
    total_words[label] += len(tokens)

word_counts

{'SPAM': Counter({'free': 2,
          'money': 1,
          'now': 1,
          'lowest': 1,
          'price': 1,
          'for': 2,
          'your': 1,
          'meds': 1,
          'win': 1,
          'a': 1,
          'iphone': 1,
          'today': 1,
          'get': 1,
          '50': 1,
          'percent': 1,
          'off': 1,
          'limited': 1,
          'time': 1,
          'click': 1,
          'here': 1,
          'prizes': 1}),
 'HAM': Counter({'hi': 1,
          'mom': 1,
          'how': 1,
          'are': 2,
          'you': 2,
          'we': 1,
          'still': 1,
          'on': 1,
          'for': 1,
          'dinner': 1,
          'lets': 1,
          'catch': 1,
          'up': 1,
          'tomorrow': 2,
          'at': 2,
          'the': 3,
          'office': 2,
          'meeting': 2,
          '3': 1,
          'pm': 1,
          'team': 1,
          'in': 1,
          'can': 1,
          'send': 1,
          'report': 1})}

In [6]:
def likelihood(word, label):
    return (word_counts[label][word] + 1) / (total_words[label] + vocab_size)

In [7]:
import math

def predict(sentence):
    tokens = tokenize(sentence)
    scores = {}

    for label in ["SPAM", "HAM"]:
        score = math.log(priors[label])
        for word in tokens:
            score += math.log(likelihood(word, label))
        scores[label] = score

    return max(scores, key=scores.get), scores

In [8]:
test_sentences = [
    "Limited offer click here",
    "Meeting at 2 PM with the manager"
]

for s in test_sentences:
    prediction, scores = predict(s)
    print(f"Sentence: {s}")
    print("Prediction:", prediction)
    print("Scores:", scores)
    print()

Sentence: Limited offer click here
Prediction: SPAM
Scores: {'SPAM': -15.587046639388863, 'HAM': -18.03297111032868}

Sentence: Meeting at 2 PM with the manager
Prediction: HAM
Scores: {'SPAM': -30.325011296597015, 'HAM': -26.8264314713814}



# Scikit-Learn Implementation

In [9]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(documents)

model = MultinomialNB()
model.fit(X, labels)

test_sentences = [
    "Limited offer click here",
    "Meeting at 2 PM with the manager"
]

X_test = vectorizer.transform(test_sentences)
predictions = model.predict(X_test)

for s, p in zip(test_sentences, predictions):
    print(f"{s} → {p}")

Limited offer click here → SPAM
Meeting at 2 PM with the manager → HAM
